In [1]:
import ee
import geemap
from dotenv import load_dotenv
import os
import ipywidgets as widgets
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd

load_dotenv(dotenv_path='.env')
project_name = os.getenv("project_name")
# print(f"Project Name: {project_name}")
ee.Initialize(project=project_name)

In [14]:
map = geemap.Map()

point = ee.Geometry.Point([90.4152, 23.8041]) #around dhaka
region = ee.Geometry.Rectangle([90.3, 23.7, 90.5, 23.9]) #bounding box around the point

top_left = [23.822424724001266, 90.46289150228976]
bottom_right = [23.796369273111445, 90.5071668854189]

#region = ee.Geometry.Rectangle([top_left[1], bottom_right[0], bottom_right[1], top_left[0]])

map.centerObject(point, 10)
map.addLayer(point, {'color': 'red'}, 'Point Layer')
map.addLayer(region, {'color': 'blue'}, 'Region Layer')

map.add_basemap('HYBRID')
map.default_style = {'cursor': 'crosshair'}


map #just testing if everything is working

Map(center=[23.8041, 90.4152], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

In [15]:
years = ['2019', '2020', '2021', '2022', '2023', '2024']
year_dropdown = widgets.Dropdown(
    options=years,
    value='2020',
    description='Select Year:',
    style={'description_width': 'initial'}
)

chart_output = widgets.Output(layout={'border': '1px solid black', 'height': '300px', 'padding': '10px'})
status_output = widgets.Output()

In [16]:
def update_map(change, region = region, map=map):
    selected_year = int(year_dropdown.value)
    
    with status_output:
        status_output.clear_output()
        print(f"laoding annual rader for the year {selected_year}")
    

    if map.find_layer('Annual Radar'):
        map.remove_layer('Annual Radar')
        
    start_date = ee.Date.fromYMD(selected_year, 1, 1)
    end_date = ee.Date.fromYMD(selected_year, 12, 31)
    

    sar_image = ee.ImageCollection('COPERNICUS/S1_GRD')\
        .filterBounds(region)\
        .filterDate(start_date, end_date)\
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
        .filter(ee.Filter.eq('instrumentMode', 'IW'))\
        .median()\
        .clip(region)
        

    vis_params = {'min': -25, 'max': -5}
    map.addLayer(sar_image, vis_params, 'Annual Radar')
    
    if not map.find_layer('Region Boundary'):
        empty = ee.Image().byte()
        outline = empty.paint(featureCollection=ee.FeatureCollection(region), color=1, width=2)
        map.addLayer(outline, {'palette': 'red'}, 'region boundary')
        
    with status_output:
        status_output.clear_output()
        print(f"desplaying {selected_year} data.")

In [17]:
year_dropdown.observe(update_map, names='value')

In [ ]:
def handle_interaction(**kwargs):
    latlon = kwargs.get('coordinates')
    
    if kwargs.get('type') == 'click':
        lat, lon = latlon
        point = ee.Geometry.Point([lon, lat])
        
        print(f"clicked at ({lat:.4f}, {lon:.4f})")

        if map.find_layer('Selected Point'):
            map.remove_layer('Selected Point')
        map.addLayer(point, {'color': 'red'}, 'Selected Point')
        
        # getting the year from dropdown
        current_year = int(year_dropdown.value)
        start_date = ee.Date.fromYMD(current_year, 1, 1)
        end_date = ee.Date.fromYMD(current_year, 12, 31)
        

        with chart_output:
            chart_output.clear_output()
            print(f"fetching for ({lat:.4f}, {lon:.4f})...")
            
        #collecting data for char
        chart_collection = ee.ImageCollection('COPERNICUS/S1_GRD') \
            .filterBounds(point) \
            .filterDate(start_date, end_date) \
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
            .filter(ee.Filter.eq('instrumentMode', 'IW')) \
            .select('VH')
            
        #getting data as a list
        def extract_values(image):
            value = image.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=point,
                scale=10
            ).get('VH')
            return ee.Feature(None, {
                'date': image.date().format('YYYY-MM-dd'),
                'value': value
            })
        
        features = chart_collection.map(extract_values)
        data = features.getInfo()
        
        # Extract dates and values
        dates = [f['properties']['date'] for f in data['features'] if f['properties']['value'] is not None]
        values = [f['properties']['value'] for f in data['features'] if f['properties']['value'] is not None]
        
        # Create the chart
        with chart_output:
            chart_output.clear_output(wait=True)
            
            if len(dates) > 0:
                fig, ax = plt.subplots(figsize=(10, 4))
                ax.plot(dates, values, marker='o', linestyle='-', markersize=3)
                ax.set_xlabel('Date')
                ax.set_ylabel('Radar Intensity (dB)')
                ax.set_title(f'Radar Intensity ({current_year}) at ({lat:.4f}, {lon:.4f})')
                ax.grid(True, alpha=0.3)
                
                # Rotate x-axis labels for better readability
                plt.xticks(rotation=45, ha='right')
                plt.tight_layout()
                plt.show()
            else:
                print("No data available for this location.")

map.on_interaction(handle_interaction)

In [19]:
update_map(None)

ui_layout = widgets.VBox([
    widgets.HTML("<h2>Radar Monitor (Annual)</h2>"),
    widgets.Label("select year to load annual radar data:", style={'font_weight': 'bold'}),
    year_dropdown,
    status_output,
    map,
    widgets.Label("results", style={'font_weight': 'bold'}),
    chart_output
])

In [20]:
ui_layout